In [1]:
import time
import pandas as pd
import numpy as np

In [2]:
df = pd.read_parquet('data/raw/full_concated_copy.parquet')
df

,EIC-код,Група,Дата,Year,Month,Day,Hour,Sum of кВт,Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год,Average of Ціна ЕЕ грн. без ПДВ/кВт*год,...,surface_pressure,pressure_msl,visibility,wind_speed_10m,wind_direction_10m,wind_gusts_10m,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance
4464,62Z0211989286227,Б,2024-12-01,2024,12,1,1,16.497020,1.46219,6.136750,...,1016.9,1036.5,None,4.2,95.0,8.6,0.0,0.0,0.0,0.0
4465,62Z0211989286227,Б,2024-12-01,2024,12,1,2,14.932647,1.46219,6.136750,...,1016.6,1036.2,None,4.5,85.0,9.0,0.0,0.0,0.0,0.0
4466,62Z0211989286227,Б,2024-12-01,2024,12,1,3,14.363785,1.46219,6.136750,...,1016.4,1035.9,None,6.0,100.0,11.9,0.0,0.0,0.0,0.0
4467,62Z0211989286227,Б,2024-12-01,2024,12,1,4,13.794922,1.46219,6.136750,...,1016.0,1035.5,None,6.4,101.0,12.6,0.0,0.0,0.0,0.0
4468,62Z0211989286227,Б,2024-12-01,2024,12,1,5,13.937137,1.46219,6.136750,...,1015.7,1035.2,None,6.4,117.0,11.9,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4362,62Z5692449931680,Б,2024-01-31,2024,1,31,20,12.619683,1.63103,3.662765,...,998.5,NaN,None,14.7,242.0,24.5,0.0,NaN,0.0,0.0
4363,62Z5692449931680,Б,2024-01-31,2024,1,31,21,12.331140,1.63103,3.662765,...,998.4,NaN,None,13.8,250.0,24.1,0.0,NaN,0.0,0.0
4364,62Z5692449931680,Б,2024-01-31,2024,1,31,22,11.568950,1.63103,3.662765,...,997.9,NaN,None,14.3,252.0,23.8,0.0,NaN,0.0,0.0
4365,62Z5692449931680,Б,2024-01-31,2024,1,31,23,10.534550,1.63103,3.662765,...,997.3,NaN,None,13.9,253.0,23.8,0.0,NaN,0.0,0.0


In [3]:
categorical_cols = ['EIC-код', 'Група', 'АЗС', 'Тип', 'Область', 'ОСР код', 'ОСР опис']
cols_to_drop = ['Дата', '__source_file', 'Унікод', 'weather_code', 'pressure_msl', 'visibility', 'direct_radiation']

In [4]:
df.drop(columns=cols_to_drop, inplace=True)

azk_cat_map = {
    "Б": 0,
    "А": 1
}
df["Група"] = df["Група"].map(azk_cat_map)

df["datetime"] = pd.to_datetime(df["datetime"])

df["day_of_week"] = df["datetime"].dt.dayofweek

month = df["datetime"].dt.month
df["season_number"] = month.map({
    12: 1, 1: 1, 2: 1,   # winter
    3: 2, 4: 2, 5: 2,    # spring
    6: 3, 7: 3, 8: 3,    # summer
    9: 4, 10: 4, 11: 4   # autumn
})

df

,EIC-код,Група,Year,Month,Day,Hour,Sum of кВт,Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год,Average of Ціна ЕЕ грн. без ПДВ/кВт*год,АЗС,...,cloud_cover_high,surface_pressure,wind_speed_10m,wind_direction_10m,wind_gusts_10m,shortwave_radiation,diffuse_radiation,direct_normal_irradiance,day_of_week,season_number
4464,62Z0211989286227,0,2024,12,1,1,16.497020,1.46219,6.136750,АЗС_23,...,0.0,1016.9,4.2,95.0,8.6,0.0,0.0,0.0,6,1
4465,62Z0211989286227,0,2024,12,1,2,14.932647,1.46219,6.136750,АЗС_23,...,0.0,1016.6,4.5,85.0,9.0,0.0,0.0,0.0,6,1
4466,62Z0211989286227,0,2024,12,1,3,14.363785,1.46219,6.136750,АЗС_23,...,0.0,1016.4,6.0,100.0,11.9,0.0,0.0,0.0,6,1
4467,62Z0211989286227,0,2024,12,1,4,13.794922,1.46219,6.136750,АЗС_23,...,0.0,1016.0,6.4,101.0,12.6,0.0,0.0,0.0,6,1
4468,62Z0211989286227,0,2024,12,1,5,13.937137,1.46219,6.136750,АЗС_23,...,0.0,1015.7,6.4,117.0,11.9,0.0,0.0,0.0,6,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4362,62Z5692449931680,0,2024,1,31,20,12.619683,1.63103,3.662765,АЗС_12,...,0.0,998.5,14.7,242.0,24.5,0.0,0.0,0.0,2,1
4363,62Z5692449931680,0,2024,1,31,21,12.331140,1.63103,3.662765,АЗС_12,...,0.0,998.4,13.8,250.0,24.1,0.0,0.0,0.0,2,1
4364,62Z5692449931680,0,2024,1,31,22,11.568950,1.63103,3.662765,АЗС_12,...,0.0,997.9,14.3,252.0,23.8,0.0,0.0,0.0,2,1
4365,62Z5692449931680,0,2024,1,31,23,10.534550,1.63103,3.662765,АЗС_12,...,0.0,997.3,13.9,253.0,23.8,0.0,0.0,0.0,2,1


In [5]:
df.to_parquet("data/silver/dataset_with_weather_time_features.parquet")

In [6]:
def categorize_variables(df, categoric_columns):
    """
    Converts specified categorical columns into dummy variables.
    NaN values are also treated as a separate category.

    :param df: pandas DataFrame
    :param categoric_columns: list of column names that need to be converted to dummy variables
    :return: tuple containing:
        - df_categorized: DataFrame with dummy variables, and dropped original categorical columns
        - mapping_dict: dictionary mapping original column names to lists of their dummy variable names
    """
    # Create dummy variables for the specified columns
    df_dummies = pd.get_dummies(df[categoric_columns], dummy_na=True)

    # Create mapping dictionary
    mapping_dict = {}
    for col in categoric_columns:
        # Find all dummy columns that start with the original column name
        dummy_cols = [dummy_col for dummy_col in df_dummies.columns if dummy_col.startswith(col + '_')]
        mapping_dict[col] = dummy_cols

    # Drop original categorical columns
    df_non_cat = df.drop(columns=categoric_columns)

    # Concatenate back non-categorical and dummy columns
    df_categorized = pd.concat([df_non_cat, df_dummies], axis=1)

    return df_categorized, mapping_dict

df, dummy_dict = categorize_variables(df, categorical_cols)
df

,Year,Month,Day,Hour,Sum of кВт,Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год,Average of Ціна ЕЕ грн. без ПДВ/кВт*год,Адреса,GPS-координати - Широта,GPS-координати - Довгота,...,ОСР опис_Ужгород,ОСР опис_Франківськ,ОСР опис_Харків,ОСР опис_Херсон,ОСР опис_Хмельницький,ОСР опис_ЦЕК,ОСР опис_Черкаси,ОСР опис_Чернівці,ОСР опис_Чернігів,ОСР опис_nan
4464,2024,12,1,1,16.497020,1.46219,6.136750,"Чернівецький р-н, с. Остриця, вул. Чернівецька...",48.279279,26.055273,...,False,False,False,False,False,False,False,True,False,False
4465,2024,12,1,2,14.932647,1.46219,6.136750,"Чернівецький р-н, с. Остриця, вул. Чернівецька...",48.279279,26.055273,...,False,False,False,False,False,False,False,True,False,False
4466,2024,12,1,3,14.363785,1.46219,6.136750,"Чернівецький р-н, с. Остриця, вул. Чернівецька...",48.279279,26.055273,...,False,False,False,False,False,False,False,True,False,False
4467,2024,12,1,4,13.794922,1.46219,6.136750,"Чернівецький р-н, с. Остриця, вул. Чернівецька...",48.279279,26.055273,...,False,False,False,False,False,False,False,True,False,False
4468,2024,12,1,5,13.937137,1.46219,6.136750,"Чернівецький р-н, с. Остриця, вул. Чернівецька...",48.279279,26.055273,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4362,2024,1,31,20,12.619683,1.63103,3.662765,"Яворівський р-н, селище Немирів, вул. Яворівсь...",50.094982,23.434658,...,False,False,False,False,False,False,False,False,False,False
4363,2024,1,31,21,12.331140,1.63103,3.662765,"Яворівський р-н, селище Немирів, вул. Яворівсь...",50.094982,23.434658,...,False,False,False,False,False,False,False,False,False,False
4364,2024,1,31,22,11.568950,1.63103,3.662765,"Яворівський р-н, селище Немирів, вул. Яворівсь...",50.094982,23.434658,...,False,False,False,False,False,False,False,False,False,False
4365,2024,1,31,23,10.534550,1.63103,3.662765,"Яворівський р-н, селище Немирів, вул. Яворівсь...",50.094982,23.434658,...,False,False,False,False,False,False,False,False,False,False


In [7]:
df.to_parquet("data/gold/full_dataset.parquet", index=False)

In [8]:
val = df[(df['datetime'] > '2025-06-30') & (df['datetime'] < '2025-08-01')]
val.drop(columns=['Адреса', 'datetime'], inplace=True)
val.to_parquet("data/gold/val.parquet", index=False)
val

C:\Users\Lev\AppData\Local\Temp\ipykernel_21932\703099120.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val.drop(columns=['Адреса', 'datetime'], inplace=True)


,Year,Month,Day,Hour,Sum of кВт,Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год,Average of Ціна ЕЕ грн. без ПДВ/кВт*год,GPS-координати - Широта,GPS-координати - Довгота,temperature_2m,...,ОСР опис_Ужгород,ОСР опис_Франківськ,ОСР опис_Харків,ОСР опис_Херсон,ОСР опис_Хмельницький,ОСР опис_ЦЕК,ОСР опис_Черкаси,ОСР опис_Чернівці,ОСР опис_Чернігів,ОСР опис_nan
1346232,2025,7,1,1,20.000000,1.67615,5.441006,48.279279,26.055273,16.7,...,False,False,False,False,False,False,False,True,False,False
1346233,2025,7,1,2,17.000000,1.67615,5.441006,48.279279,26.055273,16.4,...,False,False,False,False,False,False,False,True,False,False
1346234,2025,7,1,3,17.000000,1.67615,5.441006,48.279279,26.055273,15.5,...,False,False,False,False,False,False,False,True,False,False
1346235,2025,7,1,4,16.000000,1.67615,5.441006,48.279279,26.055273,15.1,...,False,False,False,False,False,False,False,True,False,False
1346236,2025,7,1,5,15.000000,1.67615,5.441006,48.279279,26.055273,14.6,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2947,2025,6,30,20,17.451471,0.81260,4.883898,50.403745,30.684027,17.2,...,False,False,False,False,False,False,False,False,False,False
2948,2025,6,30,21,17.339782,0.81260,4.883898,50.403745,30.684027,15.6,...,False,False,False,False,False,False,False,False,False,False
2949,2025,6,30,22,17.311859,0.81260,4.883898,50.403745,30.684027,15.2,...,False,False,False,False,False,False,False,False,False,False
2950,2025,6,30,23,16.390422,0.81260,4.883898,50.403745,30.684027,14.9,...,False,False,False,False,False,False,False,False,False,False


In [9]:
test = df[df['datetime'] > '2025-07-31']
test.drop(columns=['Адреса', 'datetime'], inplace=True)
test.to_parquet("data/gold/test.parquet", index=False)
test

C:\Users\Lev\AppData\Local\Temp\ipykernel_21932\412392032.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test.drop(columns=['Адреса', 'datetime'], inplace=True)


,Year,Month,Day,Hour,Sum of кВт,Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год,Average of Ціна ЕЕ грн. без ПДВ/кВт*год,GPS-координати - Широта,GPS-координати - Довгота,temperature_2m,...,ОСР опис_Ужгород,ОСР опис_Франківськ,ОСР опис_Харків,ОСР опис_Херсон,ОСР опис_Хмельницький,ОСР опис_ЦЕК,ОСР опис_Черкаси,ОСР опис_Чернівці,ОСР опис_Чернігів,ОСР опис_nan
1346952,2025,7,31,1,21.000000,1.67615,5.441006,48.279279,26.055273,16.4,...,False,False,False,False,False,False,False,True,False,False
1346953,2025,7,31,2,17.000000,1.67615,5.441006,48.279279,26.055273,15.8,...,False,False,False,False,False,False,False,True,False,False
1346954,2025,7,31,3,17.000000,1.67615,5.441006,48.279279,26.055273,15.2,...,False,False,False,False,False,False,False,True,False,False
1346955,2025,7,31,4,16.000000,1.67615,5.441006,48.279279,26.055273,14.9,...,False,False,False,False,False,False,False,True,False,False
1346956,2025,7,31,5,16.000000,1.67615,5.441006,48.279279,26.055273,14.6,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1483,2025,8,31,20,26.379837,0.81260,5.447321,50.403745,30.684027,25.4,...,False,False,False,False,False,False,False,False,False,False
1484,2025,8,31,21,27.803749,0.81260,5.447321,50.403745,30.684027,24.4,...,False,False,False,False,False,False,False,False,False,False
1485,2025,8,31,22,27.429035,0.81260,5.447321,50.403745,30.684027,23.8,...,False,False,False,False,False,False,False,False,False,False
1486,2025,8,31,23,25.068339,0.81260,5.447321,50.403745,30.684027,23.2,...,False,False,False,False,False,False,False,False,False,False


In [10]:
train = df[df['datetime'] < '2025-07-01']
train.drop(columns=['Адреса', 'datetime'], inplace=True)
train.to_parquet("data/gold/train.parquet", index=False)
train

C:\Users\Lev\AppData\Local\Temp\ipykernel_21932\3288085819.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train.drop(columns=['Адреса', 'datetime'], inplace=True)


,Year,Month,Day,Hour,Sum of кВт,Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год,Average of Ціна ЕЕ грн. без ПДВ/кВт*год,GPS-координати - Широта,GPS-координати - Довгота,temperature_2m,...,ОСР опис_Ужгород,ОСР опис_Франківськ,ОСР опис_Харків,ОСР опис_Херсон,ОСР опис_Хмельницький,ОСР опис_ЦЕК,ОСР опис_Черкаси,ОСР опис_Чернівці,ОСР опис_Чернігів,ОСР опис_nan
4464,2024,12,1,1,16.497020,1.46219,6.136750,48.279279,26.055273,2.3,...,False,False,False,False,False,False,False,True,False,False
4465,2024,12,1,2,14.932647,1.46219,6.136750,48.279279,26.055273,2.3,...,False,False,False,False,False,False,False,True,False,False
4466,2024,12,1,3,14.363785,1.46219,6.136750,48.279279,26.055273,2.8,...,False,False,False,False,False,False,False,True,False,False
4467,2024,12,1,4,13.794922,1.46219,6.136750,48.279279,26.055273,3.0,...,False,False,False,False,False,False,False,True,False,False
4468,2024,12,1,5,13.937137,1.46219,6.136750,48.279279,26.055273,3.0,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4362,2024,1,31,20,12.619683,1.63103,3.662765,50.094982,23.434658,1.5,...,False,False,False,False,False,False,False,False,False,False
4363,2024,1,31,21,12.331140,1.63103,3.662765,50.094982,23.434658,1.0,...,False,False,False,False,False,False,False,False,False,False
4364,2024,1,31,22,11.568950,1.63103,3.662765,50.094982,23.434658,1.4,...,False,False,False,False,False,False,False,False,False,False
4365,2024,1,31,23,10.534550,1.63103,3.662765,50.094982,23.434658,1.5,...,False,False,False,False,False,False,False,False,False,False
